In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

In [14]:
import os, requests, zipfile
from google.colab import userdata

token = userdata.get('KAGGLE_TOKEN')
headers = {"Authorization": f"Bearer {token}"}

print("Downloading...")
r = requests.get(
    "https://www.kaggle.com/api/v1/datasets/download/abhisheksjha/time-series-air-quality-data-of-india-2010-2023",
    headers=headers,
    stream=True
)

with open('/content/air_quality.zip', 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

os.makedirs('/content/air_quality', exist_ok=True)
with zipfile.ZipFile('/content/air_quality.zip', 'r') as z:
    z.extractall('/content/air_quality')

print("Done!")

Downloading...
Done!


In [15]:
final_files = {
    'Delhi':     'DL030',
    'Mumbai':    'MH015',
    'Chennai':   'TN003',
    'Hyderabad': 'TG004',
    'Bengaluru': 'KA008',
}

kaggle_dfs = {}

for city, fname in final_files.items():
    df = pd.read_csv(f'/content/air_quality/{fname}.csv')
    pm25_col = [c for c in df.columns if 'PM2.5' in c][0]
    df = df[['From Date', pm25_col]].copy()
    df.columns = ['date', 'pm25']
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).set_index('date')
    daily = df['pm25'].resample('D').mean().reset_index()
    daily.columns = ['date', 'pm25']
    daily = daily[daily['date'] >= '2019-01-01'].reset_index(drop=True)
    daily['pm25'] = daily['pm25'].interpolate(method='linear')
    kaggle_dfs[city] = daily

for city, df in kaggle_dfs.items():
    print(f"{city}: {len(df)} days | nulls: {df['pm25'].isna().sum()}")

Delhi: 1551 days | nulls: 0
Mumbai: 1381 days | nulls: 0
Chennai: 1551 days | nulls: 0
Hyderabad: 1551 days | nulls: 0
Bengaluru: 1551 days | nulls: 0


In [16]:
def run_adf(series, label):
  result = adfuller(series.dropna())
  p_value = result[1]
  stationary = 'STATIONARY' if p_value < 0.05 else 'NON-STATIONARY'
  print(f"{label}")
  print(f"  ADF stat: {result[0]:.3f} | p-value: {p_value:.4f} | {stationary}")
  print()

for city, df in kaggle_dfs.items():
    print(f"=== {city} ===")
    series = df['pm25']

    run_adf(series, "Raw series")

    first_diff = series.diff(1).dropna()
    run_adf(first_diff, "After first-order differencing (lag 1)")

    seas_diff = series.diff(365).dropna()
    run_adf(seas_diff, "After seasonal differencing (lag 365)")

=== Delhi ===
Raw series
  ADF stat: -3.344 | p-value: 0.0130 | STATIONARY

After first-order differencing (lag 1)
  ADF stat: -10.908 | p-value: 0.0000 | STATIONARY

After seasonal differencing (lag 365)
  ADF stat: -11.747 | p-value: 0.0000 | STATIONARY

=== Mumbai ===
Raw series
  ADF stat: -3.094 | p-value: 0.0270 | STATIONARY

After first-order differencing (lag 1)
  ADF stat: -16.006 | p-value: 0.0000 | STATIONARY

After seasonal differencing (lag 365)
  ADF stat: -5.168 | p-value: 0.0000 | STATIONARY

=== Chennai ===
Raw series
  ADF stat: -6.921 | p-value: 0.0000 | STATIONARY

After first-order differencing (lag 1)
  ADF stat: -14.765 | p-value: 0.0000 | STATIONARY

After seasonal differencing (lag 365)
  ADF stat: -5.617 | p-value: 0.0000 | STATIONARY

=== Hyderabad ===
Raw series
  ADF stat: -3.968 | p-value: 0.0016 | STATIONARY

After first-order differencing (lag 1)
  ADF stat: -14.301 | p-value: 0.0000 | STATIONARY

After seasonal differencing (lag 365)
  ADF stat: -10.195

In [17]:
!pip install prophet -q

In [18]:
def prepare_prophet_df(df):
  prophet_df = df[["date", "pm25"]].copy()
  prophet_df.columns = ['ds', 'y']
  prophet_df["y"] = np.log(prophet_df['y'])
  return prophet_df

test_df = prepare_prophet_df(kaggle_dfs['Delhi'])
print(test_df.head())
print(f"Columns: {test_df.columns.tolist()}")
print(f"y range: {test_df['y'].min():.2f} to {test_df['y'].max():.2f}")

          ds         y
0 2019-01-01  4.975584
1 2019-01-02  5.402396
2 2019-01-03  5.768614
3 2019-01-04  5.582007
4 2019-01-05  5.611035
Columns: ['ds', 'y']
y range: 2.12 to 6.40


In [19]:
holidays = pd.DataFrame({
    'holiday': [
        'Diwali', 'Diwali', 'Diwali', 'Diwali', 'Diwali',
        'Republic Day', 'Republic Day', 'Republic Day', 'Republic Day', 'Republic Day',
        'Holi', 'Holi', 'Holi', 'Holi', 'Holi',
        'New Year', 'New Year', 'New Year', 'New Year', 'New Year',
    ],
    'ds': pd.to_datetime([
        '2019-10-27', '2020-11-14', '2021-11-04', '2022-10-24', '2023-11-12',
        '2019-01-26', '2020-01-26', '2021-01-26', '2022-01-26', '2023-01-26',
        '2019-03-21', '2020-03-10', '2021-03-29', '2022-03-18', '2023-03-08',
        '2019-01-01', '2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01',
    ])
})

print(holidays)

         holiday         ds
0         Diwali 2019-10-27
1         Diwali 2020-11-14
2         Diwali 2021-11-04
3         Diwali 2022-10-24
4         Diwali 2023-11-12
5   Republic Day 2019-01-26
6   Republic Day 2020-01-26
7   Republic Day 2021-01-26
8   Republic Day 2022-01-26
9   Republic Day 2023-01-26
10          Holi 2019-03-21
11          Holi 2020-03-10
12          Holi 2021-03-29
13          Holi 2022-03-18
14          Holi 2023-03-08
15      New Year 2019-01-01
16      New Year 2020-01-01
17      New Year 2021-01-01
18      New Year 2022-01-01
19      New Year 2023-01-01


In [20]:
def walk_forward_splits(df, min_train_days = 365, forecast_horizon = 30, step_size = 30):
  splits = []
  n = len(df)
  start = min_train_days

  while start + step_size <= n:
    train = df.iloc[:start].copy()
    test = df.iloc[start : start + forecast_horizon].copy()
    splits.append((train,test))
    start += step_size
  return splits


In [21]:
def mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))

In [22]:
from prophet import Prophet

def evaluate_prophet(df, city, holidays):
    splits = walk_forward_splits(df, min_train_days=365, forecast_horizon=30, step_size=30)

    maes, rmses = [], []

    for train, test in splits:
        train_prophet = prepare_prophet_df(train)

        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            holidays=holidays,
            seasonality_mode='multiplicative'
        )
        model.fit(train_prophet)

        future = model.make_future_dataframe(periods=30)
        forecast = model.predict(future)

        test_forecast = forecast.tail(30)['yhat'].values

        test_forecast = np.exp(test_forecast)
        actual = test['pm25'].values

        maes.append(mae(actual, test_forecast))
        rmses.append(rmse(actual, test_forecast))

    mean_mae  = np.mean(maes)
    mean_rmse = np.mean(rmses)
    print(f"{city}: MAE={mean_mae:.2f} | RMSE={mean_rmse:.2f}")
    return mean_mae, mean_rmse

prophet_results = {}
for city, df in kaggle_dfs.items():
    prophet_results[city] = evaluate_prophet(df, city, holidays)

Delhi: MAE=37.38 | RMSE=46.51
Mumbai: MAE=17.54 | RMSE=21.29
Chennai: MAE=15.40 | RMSE=20.19
Hyderabad: MAE=12.71 | RMSE=15.11
Bengaluru: MAE=11.08 | RMSE=14.80


In [ ]:
delhi_df = kaggle_dfs["Delhi"].copy()
split_idx = int(len(delhi_df) * 0.8)

train = delhi_df.iloc[:split_idx]
test  = delhi_df.iloc[split_idx:]

train_log = np.log(train['pm25'])
test_log  = np.log(test['pm25'])

model = SARIMAX(
    train_log,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 365),
    enforce_stationarity=False,
    enforce_invertibility=False
)

result = model.fit(disp=False)
print(result.summary())

In [ ]:
pred_log = result.forecast(steps=len(test))
pred = np.exp(pred_log)

print(f"Errors: {mae(test["pm25"], pred)}, {rmse(test["pm25"], pred)}")